# Titanium Alpha -- Methodology & Results

This notebook is the self-contained walkthrough of Titanium Alpha: what we built, why each component was chosen, how the validation pipeline is structured, and what the 10-year walk-forward delivered out-of-sample. It consumes only the saved outputs under `data/outputs/` -- nothing re-runs the walk-forward backtest or the 547-config grid search, so the whole notebook executes end-to-end in under five minutes on a laptop.

**Champion config (walk-forward 10y OOS, 2016-04-19 to 2026-04-17):** Sharpe **0.766**, CAGR **13.68%**, MaxDD **-21.94%**, Beta **0.566**, Alpha **+2.57%**. `rb=15` (triweekly rebalance), `vol_target=10%`, `max_weight=min(6%, 2/N)`, HRP with Ward linkage + Ledoit-Wolf shrinkage.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from matplotlib.patches import Rectangle
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
from scipy.stats import norm
from sklearn.covariance import LedoitWolf

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

OUT = Path("../data/outputs")
assert OUT.exists(), "Run this notebook from the notebooks/ directory with data/outputs/ populated"

## 1 -- Problem: why equity selection is hard

Weak-form market efficiency (Fama, 1970) says that past prices already reflect all public information, so any predictive signal is either (a) a compensation for bearing risk you already carry or (b) a mispricing that arbitrageurs will close before you can trade on it. The literature is unforgiving: the vast majority of published anomalies vanish out-of-sample (Harvey, Liu & Zhu, 2016), and the ones that survive tend to be expensive to implement once transaction costs and liquidity constraints are applied.

Titanium Alpha does not claim a new anomaly. What it claims is **disciplined construction**: rather than running a single model on a single train/test split and reporting the best Sharpe, the system (i) uses combinatorial purged cross-validation to estimate a *distribution* of Sharpe rather than a point estimate, (ii) applies the Deflated Sharpe Ratio to adjust for multiple-testing inflation across the 547 hyperparameter configurations it explored, (iii) validates the champion on an unseen temporal holdout with `n_trials=1` to eliminate the selection-bias penalty, and (iv) reports ten full years of walk-forward equity so the reader can judge whether the alpha survived regime changes (2016 reflation, 2018 trade war, 2020 COVID drawdown, 2022 rate-hike bear, 2024-25 AI rally).

## 2 -- Data

The investable universe is 52 US large-cap equities plus SPY as the benchmark, specified in `config/tickers.json`. OHLCV is ingested from Yahoo Finance via `yf.Ticker(tkr).history()` -- the per-ticker API, not `yf.download(list)`. This matters: during Phase 7 we discovered that `yf.download()` shares session and cache state across threads, so 22 of 52 tickers in a parallel run silently received the OHLCV of adjacent tickers in `tickers.json`. The thread-safe `yf.Ticker` API fixed the data integrity issue and is now enforced by the `security-data` Claude Code subagent.

The database holds 15 years of daily OHLCV (~199 000 rows, 2011-04-25 to 2026-04-17). For the walk-forward benchmark the first 756 trading days (~3 years) are consumed as covariance lookback warmup, so the active equity curve spans exactly 10 OOS years.

In [ ]:
# Load the walk-forward equity curve -- portfolio vs SPY
equity = pl.read_parquet(OUT / "benchmark_equity.parquet")
print(f"Rows: {equity.height:>5}  |  First: {equity['date'][0]}  |  Last: {equity['date'][-1]}")
print(f"Columns: {equity.columns}")
equity.head(3)

## 3 -- CPCV: Combinatorial Purged Cross-Validation

Simple train/test splits leak information: a single model fit to pre-2020 data and evaluated post-2020 tells you about *one* period, not about robustness. Walk-forward gives one path but no distribution. Lopez de Prado's **CPCV** (Advances in Financial Machine Learning, 2018, Ch. 12) splits time into `n` equal groups, holds out `k` of them as test, and iterates over all `C(n, k)` combinations. For `n=6, k=2` that's 15 distinct train/test paths.

Two additional precautions enforce zero look-ahead:

- **Purge**: for each test block, remove from the training set any observation whose label windows overlaps the test period. We purge `h + input_size - 1 = 64` days (PatchTST forecasts `h=5` steps from `input_size=60` recent closes).
- **Embargo**: drop 10 days of training data *after* each test block, so serial correlation in features cannot leak the test outcome back into training.

The visualization below shows one of the 15 path configurations -- which blocks are test (T), which are train (.), and where the purge/embargo gaps fall.

In [ ]:
# Visualize a single CPCV path -- n=6, k=2, with purge + embargo
from itertools import combinations

N, K = 6, 2
paths = list(combinations(range(N), K))
fig, ax = plt.subplots(figsize=(10, 4))

for row, path in enumerate(paths):
    for g in range(N):
        if g in path:
            ax.add_patch(Rectangle((g, row), 1, 1, facecolor="#d9534f", edgecolor="white"))
        else:
            # purge: group before a test group (within the path)
            adjacent = any((g + 1) == p for p in path)
            trailing = any((g - 1) == p for p in path)
            if adjacent:
                ax.add_patch(Rectangle((g, row), 1, 1, facecolor="#f0ad4e", edgecolor="white"))
            elif trailing:
                ax.add_patch(Rectangle((g, row), 1, 1, facecolor="#f7e7a0", edgecolor="white"))
            else:
                ax.add_patch(Rectangle((g, row), 1, 1, facecolor="#5cb85c", edgecolor="white"))

ax.set_xlim(0, N)
ax.set_ylim(len(paths), 0)
ax.set_xticks(np.arange(N) + 0.5)
ax.set_xticklabels([f"G{g}" for g in range(N)])
ax.set_yticks(np.arange(len(paths)) + 0.5)
ax.set_yticklabels([f"path {i+1}" for i in range(len(paths))])
ax.set_title(f"CPCV($n={N}, k={K}$) -- {len(paths)} paths")
legend_colors = {"Train": "#5cb85c", "Test": "#d9534f", "Purge": "#f0ad4e", "Embargo": "#f7e7a0"}
for label, col in legend_colors.items():
    ax.bar(0, 0, color=col, label=label)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.05), ncols=4)
ax.grid(False)
plt.tight_layout()
plt.show()

## 4 -- Deflated Sharpe Ratio and the problem with 547 configs

If you try enough hyperparameter configurations, *something* will post a high Sharpe ratio purely by chance. Bailey & Lopez de Prado (2014, *The Deflated Sharpe Ratio*) quantify this: the DSR is the probability that a reported Sharpe exceeds zero *after* adjusting for the number of trials, the variance of those trials' Sharpes, and the non-normality (skew and kurtosis) of the strategy's returns.

The curve below shows the critical Sharpe threshold required to achieve a 95% DSR acceptance as a function of `n_trials` -- holding excess kurtosis at 0 and skewness at 0 for simplicity. At `n_trials=547` (our three-tier grid) the threshold climbs above the Sharpe any of our 547 configs posted on their CPCV paths -- which is why the grid search rejected every single candidate. The fix is the **holdout temporal validation** path: the champion is re-evaluated on a reserved 2-year slice the grid never saw, with `n_trials=1`, so no multiple-testing penalty applies.

In [ ]:
# DSR critical Sharpe vs n_trials (approximation -- Bailey & Lopez de Prado 2014 eq 2)
# For SR_hat to achieve DSR >= 0.95, SR_hat must exceed:
#   SR_crit = <SR>  +  sqrt(Var(SR_trials)) * ((1 - gamma) * Phi_inv(1 - 1/n) + gamma * Phi_inv(1 - 1/(n*e)))
# where gamma is Euler's constant ~0.5772
gamma = 0.5772156649
ns = np.array([1, 5, 10, 20, 50, 100, 250, 547, 1000, 5000])
mean_sr = 0.0
std_sr = 1.0  # normalized
z = (1 - gamma) * norm.ppf(1 - 1 / ns) + gamma * norm.ppf(1 - 1 / (ns * np.e))
sr_crit = mean_sr + std_sr * z

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.semilogx(ns, sr_crit, marker="o", color="#2c7fb8")
ax.axvline(547, color="#d9534f", linestyle="--", alpha=0.7, label="our grid (547 configs)")
ax.axvline(1, color="#5cb85c", linestyle="--", alpha=0.7, label="holdout ($n_{trials}=1$)")
ax.set_xlabel("Number of trials")
ax.set_ylabel("Critical (annualized) Sharpe for DSR $\\geq$ 0.95")
ax.set_title("Why 547 configs zeroed DSR acceptance (and holdout restores it)")
ax.legend()
plt.tight_layout()
plt.show()
print(f"Critical Sharpe @ n=1:   {sr_crit[0]:.3f}")
print(f"Critical Sharpe @ n=547: {sr_crit[list(ns).index(547)]:.3f}")

## 5 -- Hierarchical Risk Parity: allocation without expected returns

Classical mean-variance optimization (Markowitz, 1952) requires an estimate of expected returns. Those estimates are notoriously unstable -- small changes in the inputs produce wildly different portfolios, the so-called *error-maximization* problem. Lopez de Prado's **Hierarchical Risk Parity** (HRP; 2016) sidesteps the problem: it allocates inversely proportional to variance along a *hierarchical tree* of the correlation matrix, never asking what each asset is expected to earn.

Three choices in Titanium Alpha's HRP implementation:

1. **Ledoit-Wolf shrinkage** (Ledoit & Wolf, 2004) of the covariance matrix toward a structured target. This reduces the noise in off-diagonal correlations when the lookback (~756 days) is short relative to the number of assets (52).
2. **Ward linkage**, not the single linkage of the original HRP paper. Our CPCV-OOS grid search found Ward produces more balanced clusters on a 52-asset universe, adding ~0.03-0.05 to annualized Sharpe.
3. **Sum-preserving confidence tilt**: per-asset confidences from the agent debate bias the HRP weights toward high-conviction picks, but the multiplier is centered on the weighted-mean confidence so the total allocation does not inflate.

A small synthetic example illustrates the pipeline: 10 assets, a block-correlated return matrix, Ledoit-Wolf shrinkage, Ward linkage, and the resulting dendrogram.

In [ ]:
# Synthetic block-correlated 10-asset universe
rng = np.random.default_rng(42)
n_obs, n_assets = 400, 10
factor = rng.standard_normal((n_obs, 2))
loadings = np.vstack([np.tile([1, 0], (5, 1)), np.tile([0, 1], (5, 1))])
noise = 0.5 * rng.standard_normal((n_obs, n_assets))
returns = factor @ loadings.T + noise

# Ledoit-Wolf shrunk covariance -> correlation -> distance
lw = LedoitWolf().fit(returns)
cov = lw.covariance_
corr = cov / np.sqrt(np.outer(np.diag(cov), np.diag(cov)))
dist = np.sqrt(0.5 * (1 - np.clip(corr, -1, 1)))
np.fill_diagonal(dist, 0.0)
condensed = squareform(dist, checks=False)
Z = linkage(condensed, method="ward")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
dendrogram(Z, labels=[f"A{i}" for i in range(n_assets)], ax=axes[0], color_threshold=0.6 * max(Z[:, 2]))
axes[0].set_title("Ward linkage dendrogram (Ledoit-Wolf shrunk correlations)")
axes[0].set_ylabel("distance")

im = axes[1].imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
axes[1].set_title("Shrunk correlation matrix")
axes[1].set_xticks(range(n_assets))
axes[1].set_xticklabels([f"A{i}" for i in range(n_assets)])
axes[1].set_yticks(range(n_assets))
axes[1].set_yticklabels([f"A{i}" for i in range(n_assets)])
plt.colorbar(im, ax=axes[1], shrink=0.8)
plt.tight_layout()
plt.show()
print(f"LW shrinkage coefficient: {lw.shrinkage_:.3f}  (0 = no shrinkage, 1 = full shrinkage to scaled identity)")

## 6 -- Walk-forward results: 10 years out-of-sample

The walk-forward backtester rebalances every 15 trading days (triweekly, CPCV-OOS-validated), retrains the model every 126 days (semi-annual), and targets 10% annualized volatility *ex-ante* with a 0.5-1.0 leverage band. The portfolio starts 100% in cash; 756 days of covariance warmup are consumed before any position is opened. The active equity curve below spans **2016-04-19 to 2026-04-17** (2514 trading days, exactly 10 years).

The comparison to SPY is deliberately unflattering on absolute CAGR terms -- we pay alpha in exchange for **roughly half the market beta** (0.566) and a meaningfully lower drawdown (-21.94% vs SPY's -33% COVID low).

In [ ]:
# Load saved walk-forward results
with open(OUT / "benchmark_metrics.json") as f:
    metrics = json.load(f)

eq = equity.to_pandas().set_index("date")
eq_norm = eq / eq.iloc[0]

# Drawdown series (portfolio only)
cummax = eq_norm["portfolio_value"].cummax()
drawdown = eq_norm["portfolio_value"] / cummax - 1

# Rolling 252-day Sharpe on portfolio returns
log_ret = np.log(eq["portfolio_value"]).diff()
rolling_sharpe = (log_ret.rolling(252).mean() / log_ret.rolling(252).std()) * np.sqrt(252)

fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True, gridspec_kw={"height_ratios": [3, 1, 1.2]})
axes[0].plot(eq_norm.index, eq_norm["portfolio_value"], label="Titanium Alpha", color="#2c7fb8", linewidth=1.5)
axes[0].plot(eq_norm.index, eq_norm["benchmark_value"], label="SPY buy-and-hold", color="#555555", linewidth=1.0, alpha=0.8)
axes[0].set_ylabel("Cumulative value (normalized)")
axes[0].set_title(f"Walk-forward equity (10y OOS) -- Sharpe {metrics['sharpe_ratio']:.3f}, CAGR {metrics['cagr']*100:.2f}%, MaxDD {metrics['max_drawdown']*100:.2f}%")
axes[0].legend(loc="upper left")

axes[1].fill_between(drawdown.index, drawdown * 100, 0, color="#d9534f", alpha=0.6)
axes[1].set_ylabel("Drawdown (%)")

axes[2].plot(rolling_sharpe.index, rolling_sharpe, color="#2c7fb8")
axes[2].axhline(0, color="black", linewidth=0.5)
axes[2].set_ylabel("252d rolling Sharpe")
axes[2].set_xlabel("Date")

for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nFull metrics table:")
for k, v in metrics.items():
    print(f"  {k:<30} {v:>10.4f}")

## 7 -- Regime analysis: does the alpha survive regime shifts?

A single aggregate Sharpe can hide the truth -- a strategy that posts 0.8 over ten years might be doing 1.5 in calm periods and -0.3 in crises, or vice versa. We segment the OOS period into four regimes and compute the same metrics on each:

| Regime | Window | Macro context |
|---|---|---|
| Bull | 2016-04 -- 2019-12 | Post-election reflation, Fed gradual hikes |
| COVID + recovery | 2020-01 -- 2021-12 | Pandemic, zero-rate liquidity, growth rally |
| Bear | 2022-01 -- 2022-12 | Rate hikes, inflation shock |
| Late cycle | 2023-01 -- 2026-04 | AI rally, late-cycle consolidation |

In [ ]:
import pandas as pd

regimes = {
    "Bull 2016-19":       ("2016-04-19", "2019-12-31"),
    "COVID + recov 2020-21": ("2020-01-01", "2021-12-31"),
    "Bear 2022":          ("2022-01-01", "2022-12-31"),
    "Late cycle 2023-26": ("2023-01-01", "2026-04-17"),
}

rows = []
for name, (start, end) in regimes.items():
    mask = (eq.index >= start) & (eq.index <= end)
    seg = eq.loc[mask]
    if seg.empty:
        continue
    seg_ret_port = np.log(seg["portfolio_value"]).diff().dropna()
    seg_ret_bmk = np.log(seg["benchmark_value"]).diff().dropna()
    port_cagr = (seg["portfolio_value"].iloc[-1] / seg["portfolio_value"].iloc[0]) ** (252 / len(seg_ret_port)) - 1
    bmk_cagr = (seg["benchmark_value"].iloc[-1] / seg["benchmark_value"].iloc[0]) ** (252 / len(seg_ret_bmk)) - 1
    port_sharpe = (seg_ret_port.mean() / seg_ret_port.std()) * np.sqrt(252)
    bmk_sharpe = (seg_ret_bmk.mean() / seg_ret_bmk.std()) * np.sqrt(252)
    port_dd = (seg["portfolio_value"] / seg["portfolio_value"].cummax() - 1).min()
    rows.append({
        "Regime": name,
        "CAGR (port)": f"{port_cagr*100:.2f}%",
        "CAGR (SPY)": f"{bmk_cagr*100:.2f}%",
        "Sharpe (port)": f"{port_sharpe:.3f}",
        "Sharpe (SPY)": f"{bmk_sharpe:.3f}",
        "MaxDD (port)": f"{port_dd*100:.2f}%",
    })

regime_df = pd.DataFrame(rows)
regime_df

## 8 -- Lesson from session 40: `max_weight` is a risk constraint, not an optimizable parameter

During the Phase 8 fine-tuning we ran a PatchTST-specific grid search (session 40, `validation_6`, 435 configs). A tier-2 candidate with `rb=21` (slower rebalance) and `max_weight=0.10` (looser cap) posted a higher CPCV-OOS Sharpe than the incumbent champion (`rb=15`, `max_weight=min(6%, 2/N)=3.85%`), so we promoted it. On the walk-forward benchmark, Sharpe **collapsed to 0.462** -- a regression of 0.25 vs the 0.712 incumbent.

Why? With 52 assets, `2/N=3.85%` caps each position's contribution to portfolio risk; loosening it to 10% lets HRP concentrate ~8.5% in DUK (utilities, the lowest-variance asset in the hierarchy). Low variance wins the bisection by construction, so the cap is functional as a **risk constraint**, not a hyperparameter to minimize loss on. CPCV-OOS optimizes a single-period metric and rewarded the concentration; the walk-forward, which accumulates drawdowns through regime shifts, punished it.

The chart below reads the saved Tier 3 results and highlights the max_weight axis.

In [ ]:
# Load tier 3 results and plot Sharpe vs max_weight where it varies
tier3 = json.loads((OUT / "validation_tier3_results.json").read_text())
configs_with_mw = []
for cfg in tier3.get("configs", tier3 if isinstance(tier3, list) else []):
    if not isinstance(cfg, dict):
        continue
    params = cfg.get("params", cfg.get("config", {}))
    mw = params.get("hrp_max_weight") or params.get("max_weight")
    sharpe = cfg.get("sharpe") or cfg.get("mean_sharpe")
    if mw is not None and sharpe is not None:
        configs_with_mw.append((float(mw), float(sharpe)))

if configs_with_mw:
    mws, sharpes = zip(*configs_with_mw)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.scatter(mws, sharpes, alpha=0.6, color="#2c7fb8")
    ax.axvline(0.0385, color="#5cb85c", linestyle="--", label="champion: $\\min(6\\%, 2/N)=3.85\\%$")
    ax.axvline(0.10, color="#d9534f", linestyle="--", label="session 40 (reverted): $10\\%$")
    ax.set_xlabel("HRP max_weight cap")
    ax.set_ylabel("CPCV Sharpe (annualized)")
    ax.set_title("Tier 3: Sharpe vs max_weight -- CPCV rewards concentration that walk-forward punishes")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Tier 3 results did not expose a max_weight axis in this run; see data/outputs/validation_3tier_analysis.md for the full write-up.")

## 9 -- Limitations and next steps

Everything above validates the **walk-forward pipeline with NaiveModelFactory** (a sigmoid-based 5-day momentum score that CPCV-OOS identified as the most stable signal) plus PatchTST with model caching. The multi-agent LangGraph debate layer -- Technical / Fundamentalist / Bear / Portfolio Manager -- is **not yet backtested** with the same temporal discipline. That is the single largest gap in the project and is documented explicitly in `docs/design_gap_backtest_vs_production.md`.

Additional open items the honest reader should hold us to:

- **Stochastic debate.** The agents run at non-zero temperature (analysts 0.2, PM 0.1), so borderline decisions oscillate between BUY and HOLD across reruns. `decisions.json` is a snapshot, not an average. Averaging over N>=5 runs is the right production practice.
- **RAG grounding is bounded by news coverage.** The bootstrapped ChromaDB holds 172 articles for 52 tickers; sparse names produce ungrounded reports (`sources_cited=[]`). Production usage should run `make ingest` nightly to refresh news.
- **Long-only.** No short positions. The system goes long or flat; the cash buffer earns `rf` pro-rata.
- **No hedging.** The ~57% market beta is structural -- we do not hedge with SPY futures, volatility products, or options.
- **Universe is static.** 52 US large caps chosen at a single point in time; no survivorship correction. Forward-looking work should randomize the universe across CPCV paths.

### Next steps we would prioritize (in order)

1. **Agent-layer backtest.** Replace NaiveModelFactory with `DecisionEngine.run()` wrapped as a ModelFactory. Cache agent outputs per (ticker, date) hash so the 10-year walk-forward is tractable.
2. **Universe rotation.** Replace the static 52-ticker list with a monthly top-N-by-market-cap rebalance drawn from a larger pool (S&P 500 constituents).
3. **Short side.** Introduce a long-short variant with Bear-agent-driven short signals, then re-run the CPCV-OOS grid on the expanded search space.
4. **Options overlay.** Systematic short-dated put writes on BUY positions as an alpha source, budget-constrained by VaR.

Everything in this notebook is reproducible from the repo: `make ingest && make benchmark && jupyter nbconvert --execute notebooks/methodology_and_results.ipynb` generates the same figures from the saved outputs.